In [1]:
import requests
from bs4 import BeautifulSoup
import os
import time

# Đường dẫn thư mục gốc
base_folder_url = 'http://thpt-lequydon.edu.vn/Portals/1/thayca/HANNOM/sach/Tailieuhannomquocngu/'
# Trang chứa danh sách các liên kết (Bạn nên thay link này bằng trang mục lục nếu có)
index_url = 'http://thpt-lequydon.edu.vn/thayca/HANNOM/sach.aspx' 

headers = {'User-Agent': 'Mozilla/5.0'}

def get_all_links(url):
    try:
        response = requests.get(url, headers=headers, verify=False)
        response.encoding = response.apparent_encoding
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Tìm tất cả thẻ <a> có liên kết dẫn đến thư mục Tailieuhannomquocngu
        links = []
        for a in soup.find_all('a', href=True):
            href = a['href']
            if 'Tailieuhannomquocngu' in href and href.endswith('.htm'):
                # Chuyển link tương đối thành link tuyệt đối
                full_url = href if href.startswith('http') else base_folder_url + href.split('/')[-1]
                links.append(full_url)
        return list(set(links)) # Loại bỏ link trùng
    except Exception as e:
        print(f"Lỗi khi quét danh sách: {e}")
        return []

# Lấy danh sách link
all_pages = get_all_links(index_url)
print(f"Tìm thấy {len(all_pages)} tài liệu!")

Tìm thấy 0 tài liệu!


In [2]:
output_dir = 'TaiLieu_HanNom_Txt'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

for link in all_pages:
    try:
        name = link.split('/')[-1].replace('.htm', '.txt')
        res = requests.get(link, headers=headers, verify=False)
        res.encoding = res.apparent_encoding
        
        soup = BeautifulSoup(res.text, 'html.parser')
        # Lấy nội dung chính (bỏ script/style)
        for s in soup(['script', 'style']): s.decompose()
        clean_text = soup.get_text(separator='\n')
        
        # Lưu file
        with open(os.path.join(output_dir, name), 'w', encoding='utf-8') as f:
            f.write(f"NGUỒN: {link}\n\n")
            f.write(clean_text)
            
        print(f"Đã lưu: {name}")
        time.sleep(1) # Nghỉ 1 giây để tránh bị server chặn IP
    except:
        print(f"Lỗi khi tải: {link}")

print("--- HOÀN THÀNH ---")

--- HOÀN THÀNH ---
